In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# 1.Text Dataset (Excerpt from Tagore's Gitanjali)
data = """
Where the mind is without fear and the head is held high
Where knowledge is free
Where the world has not been broken up into fragments
By narrow domestic walls
Where words come out from the depth of truth
Where tireless striving stretches its arms towards perfection
Where the clear stream of reason has not lost its way
Into the dreary desert sand of dead habit
Where the mind is led forward by thee
Into ever-widening thought and action
Into that heaven of freedom, my Father, let my country awake.
"""

# 2. Tokenization
tokenizer = Tokenizer()
corpus = data.lower().split("\n")
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

# 3. Creating N-gram Sequences
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# 4. Padding Sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# 5. Splitting into Predictors (X) and Labels (y)
X, labels = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(labels, num_classes=total_words)

In [2]:
# 6. Building the Model
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(total_words, 10, input_length=max_sequence_len-1),
    tf.keras.layers.SimpleRNN(50),
    tf.keras.layers.Dense(total_words, activation='softmax')
])

# 7. Compiling the Model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 8. Training the Model
history = model.fit(X, y, epochs=150, verbose=1)

Epoch 1/150
3/3 [==============================] - 1s 3ms/step - loss: 4.1688 - accuracy: 0.0000e+00
Epoch 2/150
3/3 [==============================] - 0s 3ms/step - loss: 4.1459 - accuracy: 0.0125
Epoch 3/150
3/3 [==============================] - 0s 3ms/step - loss: 4.1295 - accuracy: 0.0125
Epoch 4/150
3/3 [==============================] - 0s 3ms/step - loss: 4.1127 - accuracy: 0.0875
Epoch 5/150
3/3 [==============================] - 0s 3ms/step - loss: 4.0899 - accuracy: 0.0875
Epoch 6/150
3/3 [==============================] - 0s 4ms/step - loss: 4.0643 - accuracy: 0.0875
Epoch 7/150
3/3 [==============================] - 0s 3ms/step - loss: 4.0418 - accuracy: 0.0875
Epoch 8/150
3/3 [==============================] - 0s 4ms/step - loss: 4.0177 - accuracy: 0.0875
Epoch 9/150
3/3 [==============================] - 0s 3ms/step - loss: 3.9958 - accuracy: 0.0875
Epoch 10/150
3/3 [==============================] - 0s 3ms/step - loss: 3.9797 - accuracy: 0.0875
Epoch 11/150
3/3 [=======

In [3]:
# 9. Text Generation Function
def generate_text(seed_text, next_words, model, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text

# 10. Run the Generation
print("Generated Text:")
print(generate_text("Where the mind", 5, model, max_sequence_len))

Generated Text:
Where the mind is led forward by thee
